In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 6 — Local Email Reply Assistant
## Classify Email Intent and Draft Context-Aware Replies

**Stack:** Ollama · LangChain · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from enum import Enum

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

## Step 2 — Sample Emails

In [ ]:
emails = [
    {
        "from": "client@example.com",
        "subject": "Urgent: Production API returning 500 errors",
        "body": """Hi Team,
Since 2pm today our dashboard is showing 500 errors on the /analytics endpoint.
This is blocking our quarterly reporting. Can someone look into this ASAP?
- Sarah, VP Engineering"""
    },
    {
        "from": "hr@company.com",
        "subject": "Team offsite planning",
        "body": """Hi everyone,
We're planning a team offsite for next month. Please fill out the survey by Friday
with your date preferences and activity suggestions.
Thanks, HR Team"""
    },
    {
        "from": "vendor@saas.io",
        "subject": "Your subscription renewal",
        "body": """Dear Customer,
Your annual subscription ($2,400/yr) renews on Feb 15. We've added new features
including advanced analytics and API rate limit increases. If you'd like to
discuss pricing or upgrade options, let me know.
Best, Account Manager"""
    },
]
print(f"Loaded {len(emails)} sample emails")

## Step 3 — Email Classification

In [ ]:
class EmailCategory(str, Enum):
    URGENT_ISSUE = "urgent_issue"
    ACTION_REQUIRED = "action_required"
    INFORMATION = "information"
    SALES = "sales"
    SCHEDULING = "scheduling"

class EmailClassification(BaseModel):
    category: EmailCategory
    urgency: int = Field(description="1-5 urgency level")
    sentiment: str = Field(description="positive, neutral, negative")
    key_entities: list[str] = Field(description="People, products, dates mentioned")
    requires_response: bool
    suggested_response_time: str = Field(description="e.g., 'within 1 hour', 'by Friday'")

classifier = llm.with_structured_output(EmailClassification)

for email in emails:
    print(f"\nSubject: {email['subject']}")
    result = classifier.invoke(
        f"Classify this email:\nFrom: {email['from']}\n"
        f"Subject: {email['subject']}\nBody: {email['body']}"
    )
    print(f"  Category: {result.category.value}")
    print(f"  Urgency: {result.urgency}/5")
    print(f"  Sentiment: {result.sentiment}")
    print(f"  Entities: {result.key_entities}")
    print(f"  Needs Response: {result.requires_response} ({result.suggested_response_time})")

## Step 4 — Context-Aware Reply Drafting

In [ ]:
reply_prompt = ChatPromptTemplate.from_messages([
    ("system", """You draft professional email replies. Rules:
- Match the formality level of the original email
- For urgent issues: acknowledge, provide timeline, ask clarifying questions
- For scheduling: confirm availability or suggest alternatives
- For sales: be polite but noncommittal unless clearly interested
- Keep replies concise (under 150 words)
- Include a clear next step or call to action"""),
    ("human", """Draft a reply to this email:
From: {sender}
Subject: {subject}
Body: {body}

My role: Senior Software Engineer
My context: {context}""")
])

reply_chain = reply_prompt | llm | StrOutputParser()

contexts = [
    "I'm the on-call engineer this week. I can check the logs.",
    "I'm available most dates next month except the 15th-17th.",
    "Our contract is up for review. Budget is tight this quarter.",
]

for email, ctx in zip(emails, contexts):
    print(f"\n{'='*60}")
    print(f"RE: {email['subject']}")
    print('='*60)
    reply = reply_chain.invoke({
        "sender": email["from"],
        "subject": email["subject"],
        "body": email["body"],
        "context": ctx,
    })
    print(reply)

## Step 5 — Batch Processing Pipeline

In [ ]:
def process_email(email, context=""):
    classification = classifier.invoke(
        f"From: {email['from']}\nSubject: {email['subject']}\nBody: {email['body']}"
    )
    reply = reply_chain.invoke({
        "sender": email["from"], "subject": email["subject"],
        "body": email["body"], "context": context,
    })
    return {"classification": classification, "reply": reply}

# Process all emails
results = []
for email, ctx in zip(emails, contexts):
    r = process_email(email, ctx)
    results.append(r)

# Priority-sorted output
sorted_results = sorted(results, key=lambda x: x["classification"].urgency, reverse=True)
for r in sorted_results:
    c = r["classification"]
    print(f"[Urgency {c.urgency}] {c.category.value} — Response: {c.suggested_response_time}")

## What You Learned
- **Email classification** with structured Pydantic output
- **Context-aware reply drafting** with persona and situation context
- **Priority-based batch processing** pipeline